<a href="https://colab.research.google.com/github/rolandogonzales-deeptech/Research-Methods-UNMSM-2026-II/blob/main/Deliverable6_Independent_Reproduction_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deliverable 6 — Independent reproduction of a published study
**UNMSM · Doctoral Program in Deep Tech · Reproducibility Audit**  
**Study:** Okyere & Nyanney (2026), *Stability and Interpretability of Penalized Logistic Regression Models for Breast Cancer Risk Prediction*.  
**Original source:** https://github.com/francisokyeregh/breast-cancer-penalized-logistic-regression

**Status:** This notebook is an **execution protocol**, not proof of successful reproduction. Run all cells in a fresh **Google Colab CPU** runtime, inspect the log, and retain outputs. The author's R script is preserved; only its machine-specific absolute directory is replaced in a separate copy. The original script has a hard-coded Windows path and its actual package list is broader than the README's list. A further methodological audit is required before interpreting performance.


## 1. Download original repository and record its commit

In [1]:
import os, subprocess, pathlib, hashlib, json, platform, datetime
root=pathlib.Path('/content/repro_ookyere')
if root.exists():
    raise RuntimeError('Directory already exists. Use a fresh runtime to preserve provenance.')
subprocess.run(['git','clone','https://github.com/francisokyeregh/breast-cancer-penalized-logistic-regression.git',str(root)],check=True)
commit=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip()
source=root/'R Script.R'; data=root/'data.csv'
assert source.is_file() and data.is_file(), 'Missing original script or data.csv'
sha=lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
metadata={'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(), 'commit':commit,'original_script_sha256':sha(source),'data_sha256':sha(data),'python':platform.python_version(),'platform':platform.platform()}
(root/'audit_metadata.json').write_text(json.dumps(metadata,indent=2))
print(json.dumps(metadata,indent=2))


{
  "timestamp_utc": "2026-10-18T05:11:26.942990+00:00",
  "commit": "788f6280c2bc823d55885b0044ba18a564e24ef1",
  "original_script_sha256": "2eabc8510fad73825caef2ea124e51b7ebc671ac7b8405407ac352f2189679f9",
  "data_sha256": "1425d9affa78ba8e53afc81d0ef8a19069ee10c4b21fe89b3cf514071b12ee33",
  "python": "3.13.16",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39"
}


## 2. Install R and required packages (may take several minutes)

In [2]:
import subprocess
subprocess.run(['bash','-lc','apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev'],check=True)
subprocess.run(['Rscript','-e','cat(R.version.string,"\n")'],check=True)


CompletedProcess(args=['Rscript', '-e', 'cat(R.version.string,"\n")'], returncode=0)

In [3]:
import subprocess
pkgs=['tidyverse','glmnet','pROC','rsample','yardstick','broom','fs','glue','forcats','patchwork']
r='options(repos=c(CRAN="https://cloud.r-project.org")); needed <- '+ 'c('+','.join('"'+p+'"' for p in pkgs)+'); missing <- setdiff(needed, rownames(installed.packages())); if(length(missing)) install.packages(missing, Ncpus=2); stopifnot(all(needed %in% rownames(installed.packages())))'
subprocess.run(['Rscript','-e',r],check=True)


CompletedProcess(args=['Rscript', '-e', 'options(repos=c(CRAN="https://cloud.r-project.org")); needed <- c("tidyverse","glmnet","pROC","rsample","yardstick","broom","fs","glue","forcats","patchwork"); missing <- setdiff(needed, rownames(installed.packages())); if(length(missing)) install.packages(missing, Ncpus=2); stopifnot(all(needed %in% rownames(installed.packages())))'], returncode=0)

## 3. Adapt the Windows-only path without changing the scientific logic

In [4]:
original=source.read_text(encoding='utf-8-sig')
old='ROOT <- "C:/Users/kokye/OneDrive/Desktop/Research1"'
assert original.count(old)==1, 'Upstream script changed; inspect path manually before running.'
adapted=original.replace(old, f'ROOT <- "{root}"')
run_script=root/'R_Script_colab_path_only.R'
run_script.write_text(adapted,encoding='utf-8')
print('Original SHA-256:',sha(source))
print('Adapted SHA-256: ',sha(run_script))
print('Only substitution:',old,'→',f'ROOT <- "{root}"')


Original SHA-256: 2eabc8510fad73825caef2ea124e51b7ebc671ac7b8405407ac352f2189679f9
Adapted SHA-256:  5f0bdc0d6aceb3a024442a32361bd20190c75b7949a1ebc0b912c3b3807a1017
Only substitution: ROOT <- "C:/Users/kokye/OneDrive/Desktop/Research1" → ROOT <- "/content/repro_ookyere"


## 4. Execute the original analysis (with path substitution) and preserve stdout/stderr

In [5]:
import subprocess, time
start=time.monotonic()
log=root/'reproduction_run.log'
with log.open('w') as f:
    proc=subprocess.run(['Rscript',str(run_script)],cwd=root,stdout=f,stderr=subprocess.STDOUT,timeout=3600)
print('Return code:',proc.returncode,'| Runtime seconds:',round(time.monotonic()-start,1))
print('Log:',log)
print('Last 5000 characters of log:')
print(log.read_text(errors='replace')[-5000:])
if proc.returncode != 0:
    print('EXECUTION FAILED: preserve the log and report the error; do not claim reproduction.')


Return code: 0 | Runtime seconds: 34.3
Log: /content/repro_ookyere/reproduction_run.log
Last 5000 characters of log:
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: Matrix

Attaching package: ‘Matrix’

The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack

Loaded glmnet 5.1
Type 'citation("pROC")' for a citation.

Attaching package: ‘pROC’

The following objects are masked from ‘package:stats’:

    cov, smooth, var


Attaching package: ‘yardstick’

The following obj

## 5. Inventory results, record software versions, and flag potential test-set leakage

In [6]:
import pathlib, subprocess
results=root/'results'
files=sorted(p for p in results.rglob('*') if p.is_file()) if results.exists() else []
for p in files[:120]: print(p.relative_to(root),p.stat().st_size,'bytes')
print('Total result files:',len(files))
print('R environment:')
subprocess.run(['Rscript','-e','sessionInfo()'])
print('AUDIT WARNING: upstream code computes Youden threshold on y_test / p_glm (test labels).')
print('This can bias threshold-optimized test performance. Treat fixed 0.5-threshold metrics separately.')


results/figures/Figure3_Stability_Combined.pdf 6532 bytes
results/figures/Figure3_Stability_Combined.png 380985 bytes
results/figures/calibration_best_model.png 123642 bytes
results/figures/coef_paths_merged_lasso_enet.pdf 17826 bytes
results/figures/coef_paths_merged_lasso_enet.png 194477 bytes
results/figures/roc_all_models.png 131111 bytes
results/figures/roc_best_model.png 101654 bytes
results/rds/cv_glmnet_enet_alpha0.5.rds 23642 bytes
results/rds/cv_glmnet_lasso_alpha1.rds 17313 bytes
results/rds/cv_glmnet_ridge_alpha0.rds 31018 bytes
results/rds/glm_unpenalized.rds 239164 bytes
results/rds/scaling_train_only.rds 839 bytes
results/sessionInfo.txt 2147 bytes
results/tables/metrics_enet_threshold_0_5.csv 159 bytes
results/tables/metrics_enet_threshold_opt_youden.csv 173 bytes
results/tables/metrics_glm_threshold_0_5.csv 175 bytes
results/tables/metrics_glm_threshold_opt_youden.csv 176 bytes
results/tables/metrics_lasso_threshold_0_5.csv 176 bytes
results/tables/metrics_lasso_thresh

## 6. Inspect numeric outputs and compare with the article

In [7]:
import pandas as pd
csvs=sorted((root/'results'/'tables').glob('*.csv')) if (root/'results'/'tables').exists() else []
for p in csvs:
    print('\n',p.name)
    try: display(pd.read_csv(p).head(12))
    except Exception as exc: print('Unable to parse:',exc)
print('Compare each metric to the published article manually; record both numbers, the exact table/figure, and any discrepancy.')



 metrics_enet_threshold_0_5.csv


,threshold,accuracy,sensitivity,specificity,auc,brier,tp,tn,fp,fn
0,0.5,0.982609,1,0.972222,0.999354,0.012132,43,70,2,0



 metrics_enet_threshold_opt_youden.csv


,threshold,accuracy,sensitivity,specificity,auc,brier,tp,tn,fp,fn
0,0.521872,0.991304,1,0.986111,0.999354,0.012132,43,71,1,0



 metrics_glm_threshold_0_5.csv


...


 stability_ridge.csv


,feature,selection_frequency,alpha,lambda_used,B
0,radius_mean,1,0,lambda.min,1000
1,texture_mean,1,0,lambda.min,1000
2,perimeter_mean,1,0,lambda.min,1000
3,area_mean,1,0,lambda.min,1000
4,smoothness_mean,1,0,lambda.min,1000
5,compactness_mean,1,0,lambda.min,1000
6,concavity_mean,1,0,lambda.min,1000
7,concave.points_mean,1,0,lambda.min,1000
8,symmetry_mean,1,0,lambda.min,1000
9,fractal_dimension_mean,1,0,lambda.min,1000



 stable_features_pi_0.7.csv


,model,n_stable,stable_features
0,Ridge,30,"radius_mean, texture_mean, perimeter_mean, are..."
1,Lasso,9,"texture_worst, radius_se, radius_worst, smooth..."
2,ElasticNet,22,"radius_se, area_se, radius_worst, texture_wors..."


Compare each metric to the published article manually; record both numbers, the exact table/figure, and any discrepancy.


## 7. Evidence-based audit — fill in after execution

| Criterion | Rating (Yes / Partial / No) | Evidence from code + run |
|---|---|---|
| Seeds and multiple runs | Pending | `set.seed(42)` observed; verify repetitions and variability |
| Splits described | Pending | Stratified 80/20; verify split in outputs |
| Leakage checked | Pending | Training-only scaling; **test-set Youden threshold selection** needs attention |
| Tuning separated | Pending | `cv.glmnet` on training data; inspect remaining code |
| Statistical test named | Pending | Inspect article and code |
| Effect size and confidence intervals | Pending | Inspect article and code |
| External validation | Pending | Inspect study design |
| Compute documented | Pending | `sessionInfo()` and timing; hardware details |
| Code and data available | Yes | Original GitHub repo, script and `data.csv` |

**Interpretation rule:** A successful script run is *computational executability*, not necessarily exact numerical reproduction or methodological validity. Do not claim the scorecard is final without evidence.

**Important source-code concern:** the original script's baseline logistic regression obtains its optimized classification threshold using the **test-set outcomes**. If the paper reports performance at this optimized threshold as unbiased test performance, that estimate may be optimistic. This observation should be discussed, not silently corrected during the reproduction.


## 8. Download all evidence (logs, data hashes, generated results)

In [8]:
import zipfile
archive=pathlib.Path('/content/Deliverable6_Independent_Reproduction_Evidence.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob('*'):
        if p.is_file() and '.git' not in p.parts: z.write(p,p.relative_to(root.parent))
print(archive,archive.stat().st_size,'bytes')
from google.colab import files
files.download(str(archive))


/content/Deliverable6_Independent_Reproduction_Evidence.zip 2767531 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>